# arynews.tv Urdu News Scraper

**Goal**: Scrape all Urdu articles from `https://urdu.arynews.tv/` across 10 categories for the Urdu NLP corpus.

**Architecture**:
- Phase 1: Discover article URLs by walking category pagination (concurrent)
- Phase 2: Fetch articles concurrently (16 workers), extract structured fields
- Save to Google Drive as gzip JSONL shards
- Checkpoint every 30 minutes — if Colab disconnects, just re-run the cell and it resumes

**Output per article**: `{url, title, category, published_date, body_text, char_count, scraped_at}`

**Categories scraped** (multimedia excluded — video/photo gallery only, no article text):
1. پاکستان (Pakistan)
2. عالمی خبریں (World)
3. کھیل (Sports)
4. حیرت انگیز (Amazing)
5. تجارت (Business)
6. صحت (Health)
7. فن و ثقافت (Arts & Culture)
8. میگزین (Magazine)
9. سائنس اور ٹیکنالوجی (Science & Tech)
10. بلاگز (Blogs)

**How to use**:
1. Run Cell 1 (install dependencies) — ~30 sec
2. Run Cell 2 (mount Drive) — follow the auth prompt
3. Run Cell 3 (config + scraper code) — defines everything, no execution
4. Run Cell 4 (start scraping) — leave running. If disconnected, just re-run Cell 4 to resume.
5. Run Cell 5 (verify output) — when scraping is done or to check progress.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q requests beautifulsoup4 lxml tqdm
print('Dependencies installed.')

## Cell 2 — Mount Google Drive

Click the URL that appears, sign in with the same Google account that owns your Drive, paste the auth code back.

All scraped data will be saved to `MyDrive/urdu_corpus/arynews/` so it persists across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/urdu_corpus/arynews'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive mounted. Output will be saved to: {DRIVE_BASE}')

## Cell 3 — Scraper Code (Configuration + All Functions)

This cell defines everything but does NOT start scraping. Run it once to load all functions into memory.

In [ ]:
# ============================================================
# arynews.tv Urdu News Scraper — Configuration & Functions
# ============================================================

import os, sys, json, gzip, time, random, logging, threading
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ---------- CONFIG ----------
BASE_URL = 'https://urdu.arynews.tv'

CATEGORIES = {
    'pakistan-2':              'پاکستان',
    'international-2':         'عالمی خبریں',
    'sports-2':                'کھیل',
    'unsual':                  'حیرت انگیز',
    'کاروباری-خبریں':          'تجارت',
    'sehat':                   'صحت',
    'fun-o-sakafat':           'فن و ثقافت',
    'miscellaneous':           'میگزین',
    'سائنس-اور-ٹیکنالوجی':     'سائنس اور ٹیکنالوجی',
    'urdu-blogs':              'بلاگز',
}

WORKERS = 16                       # concurrent threads for article fetching
PAGE_WORKERS = 8                   # concurrent threads for URL discovery
REQUEST_TIMEOUT = 20
MAX_RETRIES = 4
RETRY_BACKOFF = [2, 5, 15, 30]
RATE_LIMIT_DELAY = (0.05, 0.15)    # random sleep between requests

USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Version/17.0 Safari/605.1.15',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/119.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0',
]

CHECKPOINT_INTERVAL_SECONDS = 5 * 60    # save state at least every 5 minutes
CHECKPOINT_INTERVAL_ARTICLES = 500      # ...or every 500 articles, whichever first
MAX_PAGES_PER_CATEGORY = 5000           # safety cap
PAGES_PER_SHARD = 500                   # articles per JSONL shard
MONITOR_INTERVAL_SECONDS = 15           # how often to print live status

# Output paths (on Google Drive)
OUTPUT_DIR = Path('/content/drive/MyDrive/urdu_corpus/arynews')
URLS_DIR = OUTPUT_DIR / 'urls'
ARTICLES_DIR = OUTPUT_DIR / 'articles'
LOGS_DIR = OUTPUT_DIR / 'logs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoint'

# ---------- LOGGING ----------
def setup_logging():
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    log_file = LOGS_DIR / f"scrape_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    # Remove old handlers (for re-runs in same kernel)
    root = logging.getLogger()
    for h in list(root.handlers): root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s [%(levelname)s] %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler(),
        ],
    )
    return logging.getLogger('arynews')

log = logging.getLogger('arynews')

# ---------- HTTP CLIENT ----------
class HttpClient:
    def __init__(self):
        self.session = requests.Session()

    def get(self, url):
        for attempt in range(MAX_RETRIES):
            try:
                headers = {'User-Agent': random.choice(USER_AGENTS)}
                time.sleep(random.uniform(*RATE_LIMIT_DELAY))
                r = self.session.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
                if r.status_code == 200:
                    return r.text
                elif r.status_code in (429, 503):
                    wait = RETRY_BACKOFF[min(attempt, len(RETRY_BACKOFF) - 1)] * 2
                    log.warning(f'HTTP {r.status_code} on {url}, backing off {wait}s')
                    time.sleep(wait)
                elif r.status_code == 404:
                    return None
                else:
                    log.warning(f'HTTP {r.status_code} on {url}')
                    time.sleep(RETRY_BACKOFF[min(attempt, len(RETRY_BACKOFF) - 1)])
            except requests.RequestException as e:
                wait = RETRY_BACKOFF[min(attempt, len(RETRY_BACKOFF) - 1)]
                log.warning(f'Attempt {attempt+1} failed for {url}: {type(e).__name__}')
                time.sleep(wait)
        log.error(f'Giving up on {url} after {MAX_RETRIES} attempts')
        return None

http = HttpClient()

# ---------- URL DISCOVERY ----------
def extract_article_urls_from_category_page(html):
    if not html:
        return []
    soup = BeautifulSoup(html, 'lxml')
    urls = []
    for h3 in soup.find_all('h3', class_='entry-title'):
        a = h3.find('a', href=True)
        if a:
            href = a['href']
            if (href.startswith(BASE_URL + '/')
                and '/category/' not in href
                and '/page/' not in href
                and '/tag/' not in href
                and '/author/' not in href
                and '?s=' not in href
                and 'wp-' not in href):
                path = urlparse(href).path.strip('/')
                if '/' not in path and len(path) > 5:
                    urls.append(href)
    seen = set()
    unique = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            unique.append(u)
    return unique

def discover_urls_for_category(category_slug, category_name):
    log.info(f'[DISCOVERY] Starting category: {category_name} ({category_slug})')
    discovered = []
    seen_urls = set()
    empty_streak = 0

    page_urls = []
    for page_num in range(1, MAX_PAGES_PER_CATEGORY + 1):
        if page_num == 1:
            page_urls.append((1, f'{BASE_URL}/category/{category_slug}/'))
        else:
            page_urls.append((page_num, f'{BASE_URL}/category/{category_slug}/page/{page_num}/'))

    BATCH_SIZE = 20
    with tqdm(total=min(MAX_PAGES_PER_CATEGORY, len(page_urls)),
              desc=f'  {category_name}', unit='pg') as pbar:
        for batch_start in range(0, len(page_urls), BATCH_SIZE):
            batch = page_urls[batch_start:batch_start + BATCH_SIZE]
            with ThreadPoolExecutor(max_workers=PAGE_WORKERS) as ex:
                futures = {ex.submit(http.get, url): (pnum, url) for pnum, url in batch}
                results = {}
                for fut in as_completed(futures):
                    pnum, url = futures[fut]
                    results[pnum] = fut.result()

            for pnum, url in batch:
                html = results.get(pnum)
                if html is None:
                    empty_streak += 1
                    pbar.update(1)
                    continue
                page_urls_found = extract_article_urls_from_category_page(html)
                if not page_urls_found:
                    empty_streak += 1
                    pbar.update(1)
                    continue
                new_on_page = [u for u in page_urls_found if u not in seen_urls]
                if not new_on_page:
                    empty_streak += 1
                else:
                    empty_streak = 0
                    for u in new_on_page:
                        seen_urls.add(u)
                        discovered.append({'url': u, 'category_slug': category_slug, 'category_name': category_name})
                pbar.update(1)

            if empty_streak >= 5:
                log.info(f'  [DISCOVERY] {category_name}: stopped at page {batch_start + len(batch)} (archive end)')
                break

    log.info(f'[DISCOVERY] {category_name}: found {len(discovered)} article URLs')
    return discovered

def discover_all_urls():
    URLS_DIR.mkdir(parents=True, exist_ok=True)
    all_urls = []
    for cat_slug, cat_name in CATEGORIES.items():
        urls_file = URLS_DIR / f'urls_{cat_slug}.jsonl'
        if urls_file.exists():
            log.info(f'[DISCOVERY] {cat_name}: URLs file exists, loading')
            with open(urls_file, encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        all_urls.append(json.loads(line))
            continue
        urls = discover_urls_for_category(cat_slug, cat_name)
        with open(urls_file, 'w', encoding='utf-8') as f:
            for r in urls:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')
        all_urls.extend(urls)
    log.info(f'[DISCOVERY] TOTAL: {len(all_urls)} article URLs across all categories')
    return all_urls

# ---------- ARTICLE EXTRACTION ----------
def extract_article(html, url, category_name):
    if not html:
        return None
    soup = BeautifulSoup(html, 'lxml')

    # Title: <h1 class="tdb-title-text"> is the actual article title
    title = None
    h1 = soup.find('h1', class_='tdb-title-text')
    if h1:
        title = h1.get_text(strip=True)
    else:
        article_tag = soup.find('article')
        if article_tag:
            h1 = article_tag.find('h1')
            if h1:
                title = h1.get_text(strip=True)
    if not title:
        return None

    # Body: <div class="td-post-content">
    body = None
    body_div = soup.find('div', class_='td-post-content')
    if body_div:
        for junk in body_div.find_all(class_=['code-block', 'td_block_template', 'related', 'share', 'wp-post-navigation']):
            junk.decompose()
        body = body_div.get_text('\n', strip=True)
    if not body or len(body) < 100:
        for cls in ['tdb-block-inner', 'entry-content']:
            alt = soup.find(class_=cls)
            if alt:
                txt = alt.get_text('\n', strip=True)
                if txt and len(txt) > len(body or ''):
                    body = txt
    if not body or len(body) < 100:
        return None

    # Date
    published = None
    time_tag = soup.find('time', class_='entry-date')
    if time_tag and time_tag.get('datetime'):
        published = time_tag['datetime']
    else:
        time_tag = soup.find('time')
        if time_tag and time_tag.get('datetime'):
            published = time_tag['datetime']

    return {
        'url': url,
        'title': title,
        'category': category_name,
        'published_date': published,
        'body_text': body,
        'char_count': len(body),
        'scraped_at': datetime.now(timezone.utc).isoformat(),
    }

def fetch_one_article(url, category_name):
    html = http.get(url)
    if html is None:
        return None
    return extract_article(html, url, category_name)

# ---------- CHECKPOINT / RESUME ----------
def get_checkpoint_file():
    return CHECKPOINT_DIR / 'progress.json'

def load_checkpoint():
    checkpoint_file = get_checkpoint_file()
    if checkpoint_file.exists():
        with open(checkpoint_file, encoding='utf-8') as f:
            data = json.load(f)
        return {
            'completed_urls': set(data.get('completed_urls', [])),
            'failed_urls': set(data.get('failed_urls', [])),
            'total_saved': data.get('total_saved', 0),
            'current_shard_idx': data.get('current_shard_idx', 0),
            'current_shard_count': data.get('current_shard_count', 0),
            'start_time': data.get('start_time', datetime.now(timezone.utc).isoformat()),
        }
    return {
        'completed_urls': set(),
        'failed_urls': set(),
        'total_saved': 0,
        'current_shard_idx': 0,
        'current_shard_count': 0,
        'start_time': datetime.now(timezone.utc).isoformat(),
    }

def save_checkpoint(state):
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_file = get_checkpoint_file()
    data = {
        'completed_urls': list(state['completed_urls']),
        'failed_urls': list(state['failed_urls']),
        'total_saved': state['total_saved'],
        'current_shard_idx': state['current_shard_idx'],
        'current_shard_count': state['current_shard_count'],
        'start_time': state['start_time'],
        'last_saved_at': datetime.now(timezone.utc).isoformat(),
    }
    tmp = checkpoint_file.with_suffix('.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    tmp.replace(checkpoint_file)

# ---------- ARTICLE FETCH LOOP (with checkpointing + live monitor) ----------
def get_current_shard_path(state):
    return ARTICLES_DIR / f"articles_{state['current_shard_idx']:04d}.jsonl.gz"

def append_to_shard(state, article):
    ARTICLES_DIR.mkdir(parents=True, exist_ok=True)
    shard_path = get_current_shard_path(state)
    with gzip.open(shard_path, 'at', encoding='utf-8') as f:
        f.write(json.dumps(article, ensure_ascii=False) + '\n')
    state['current_shard_count'] += 1
    state['total_saved'] += 1
    if state['current_shard_count'] >= PAGES_PER_SHARD:
        state['current_shard_idx'] += 1
        state['current_shard_count'] = 0

def format_elapsed(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

def read_last_checkpoint_time():
    cp_file = get_checkpoint_file()
    if not cp_file.exists():
        return None
    try:
        with open(cp_file, encoding='utf-8') as f:
            data = json.load(f)
        last = data.get('last_saved_at')
        if last:
            return datetime.fromisoformat(last.replace('Z', '+00:00'))
    except Exception:
        pass
    return None

def stats_monitor(state, fetch_start_time, total_pending, stop_event):
    '''Background thread: prints live stats while scraping.'''
    while not stop_event.is_set():
        elapsed = time.time() - fetch_start_time
        saved = state['total_saved']
        failed = len(state['failed_urls'])
        completed = len(state['completed_urls'])
        rate = saved / max(elapsed, 1)
        last_cp = read_last_checkpoint_time()
        if last_cp:
            since_cp = (datetime.now(timezone.utc) - last_cp).total_seconds()
            cp_str = f'{format_elapsed(since_cp)} ago'
        else:
            cp_str = 'not yet'
        remaining = total_pending - completed
        eta_str = format_elapsed(remaining / rate) if rate > 0.1 else '??'
        line = (
            f'\r[{format_elapsed(elapsed)}] '
            f'Saved: {saved:,} | '
            f'Failed: {failed:,} | '
            f'Rate: {rate:.1f}/s | '
            f'Last Drive save: {cp_str} | '
            f'ETA: {eta_str}   '
        )
        sys.stdout.write(line)
        sys.stdout.flush()
        stop_event.wait(MONITOR_INTERVAL_SECONDS)
    elapsed = time.time() - fetch_start_time
    saved = state['total_saved']
    failed = len(state['failed_urls'])
    rate = saved / max(elapsed, 1)
    sys.stdout.write('\r' + ' ' * 100 + '\r')
    sys.stdout.flush()
    log.info(
        f'[FINAL] {format_elapsed(elapsed)} elapsed | '
        f'{saved:,} saved | {failed:,} failed | '
        f'{rate:.1f} articles/sec avg'
    )

def fetch_all_articles(url_records):
    state = load_checkpoint()
    log.info(f'[FETCH] Resuming: {len(state["completed_urls"])} already done, '
             f'{len(url_records) - len(state["completed_urls"])} remaining')
    pending = [r for r in url_records if r['url'] not in state['completed_urls']]
    log.info(f'[FETCH] Pending: {len(pending)} articles')
    if not pending:
        log.info('[FETCH] Nothing to do — all articles already scraped.')
        return state

    last_checkpoint = time.time()
    articles_since_checkpoint = 0
    lock = threading.Lock()

    # Start live monitor in background
    fetch_start_time = time.time()
    stop_event = threading.Event()
    monitor_thread = threading.Thread(
        target=stats_monitor,
        args=(state, fetch_start_time, len(pending), stop_event),
        daemon=True,
    )
    monitor_thread.start()

    with tqdm(total=len(pending), desc='Articles', unit='art') as pbar:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            futures = {ex.submit(fetch_one_article, r['url'], r['category_name']): r for r in pending}
            for fut in as_completed(futures):
                r = futures[fut]
                url = r['url']
                try:
                    article = fut.result()
                except Exception as e:
                    log.error(f'Exception on {url}: {type(e).__name__}: {e}')
                    article = None
                with lock:
                    if article is not None:
                        append_to_shard(state, article)
                        state['completed_urls'].add(url)
                    else:
                        state['failed_urls'].add(url)
                    now = time.time()
                    articles_since_checkpoint += 1
                    if (now - last_checkpoint > CHECKPOINT_INTERVAL_SECONDS
                        or articles_since_checkpoint >= CHECKPOINT_INTERVAL_ARTICLES):
                        save_checkpoint(state)
                        last_checkpoint = now
                        articles_since_checkpoint = 0
                        log.info(
                            f'[CHECKPOINT] Saved to Drive. '
                            f'Total: {state["total_saved"]:,} articles, '
                            f'{len(state["failed_urls"]):,} failed'
                        )
                pbar.update(1)
                pbar.set_postfix(saved=state['total_saved'], failed=len(state['failed_urls']))

    stop_event.set()
    monitor_thread.join(timeout=5)

    save_checkpoint(state)
    log.info(f'[FETCH] DONE. Total saved: {state["total_saved"]:,}, '
             f'failed: {len(state["failed_urls"]):,}')
    return state

# ---------- STATS ----------
def print_summary(state):
    log.info('=' * 60)
    log.info('SCRAPING SUMMARY')
    log.info('=' * 60)
    log.info(f'Total articles saved:  {state["total_saved"]}')
    log.info(f'Failed URLs:           {len(state["failed_urls"])}')
    log.info(f'Started at:            {state["start_time"]}')
    log.info('')
    log.info('Per-shard file sizes:')
    for shard in sorted(ARTICLES_DIR.glob('articles_*.jsonl.gz')):
        size_mb = shard.stat().st_size / 1024 / 1024
        log.info(f'  {shard.name}: {size_mb:.2f} MB')
    log.info('')
    log.info(f'Output directory: {OUTPUT_DIR}')
    log.info('=' * 60)

# ---------- MAIN ----------
def main():
    for d in [OUTPUT_DIR, URLS_DIR, ARTICLES_DIR, LOGS_DIR, CHECKPOINT_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    setup_logging()
    log.info('=' * 60)
    log.info('ARYNEWS.tv URDU SCRAPER')
    log.info('=' * 60)
    log.info(f'Categories: {len(CATEGORIES)}')
    log.info(f'Workers: {WORKERS} (articles), {PAGE_WORKERS} (discovery)')
    log.info(f'Checkpoint interval: {CHECKPOINT_INTERVAL_SECONDS}s')
    log.info(f'Output: {OUTPUT_DIR}')
    log.info('=' * 60)

    log.info('\n>>> PHASE 1: URL DISCOVERY <<<')
    url_records = discover_all_urls()

    log.info('\n>>> PHASE 2: ARTICLE FETCHING <<<')
    state = fetch_all_articles(url_records)

    print_summary(state)

print('Scraper module loaded. Run the next cell to start scraping.')

## Cell 4 — Start Scraping (with Live Status)

**Run this cell to begin scraping.** Leave the Colab tab open.

While scraping, you'll see a live status line that updates every 15 seconds:
```
[01:23:45] Saved: 12,450 | Failed: 3 | Rate: 14.9/s | Last Drive save: 00:01:23 ago | ETA: 02:14:00
```

**Meaning of each field:**
- `[01:23:45]` — elapsed time since this scrape session started
- `Saved` — total articles successfully written to Drive
- `Failed` — articles that failed after retries (will be retried next session)
- `Rate` — articles per second (averaged since start)
- `Last Drive save` — how recently checkpoint was saved to Drive (so you know how much data is at risk if Colab dies)
- `ETA` — estimated time remaining at current rate

**Checkpoint frequency** (when progress gets saved to Drive):
- Every **5 minutes** OR every **500 articles**, whichever comes first
- Plus a final save when scraping ends
- So worst case on disconnect: you lose the last 5 minutes / 500 articles

**If Colab disconnects** (session timeout, internet blip, etc.):
1. Reconnect to Colab (refresh page)
2. Re-run Cell 1 (deps) and Cell 2 (mount Drive)
3. Re-run Cell 3 (load scraper code)
4. Re-run this cell — it will load the checkpoint and resume from where it stopped

Progress is also visible in `MyDrive/urdu_corpus/arynews/logs/` and `MyDrive/urdu_corpus/arynews/checkpoint/progress.json`.

In [ ]:
main()

# === POST-SCRAPE SUMMARY (auto-runs after main() finishes) ===
import gzip, json
from pathlib import Path

articles_dir = Path('/content/drive/MyDrive/urdu_corpus/arynews/articles')
checkpoint_file = Path('/content/drive/MyDrive/urdu_corpus/arynews/checkpoint/progress.json')

print('\n' + '='*60)
print('FINAL OUTPUT SUMMARY')
print('='*60)

# Checkpoint stats
if checkpoint_file.exists():
    with open(checkpoint_file) as f:
        cp = json.load(f)
    print(f'Total saved:     {cp["total_saved"]:,}')
    print(f'Completed URLs:  {len(cp["completed_urls"]):,}')
    print(f'Failed URLs:     {len(cp["failed_urls"]):,}')
    print(f'Started:         {cp["start_time"]}')
    print(f'Last checkpoint: {cp.get("last_saved_at", "n/a")}')

# Shard sizes
print('\nShard files on Drive:')
if articles_dir.exists():
    shards = sorted(articles_dir.glob('articles_*.jsonl.gz'))
    total_size = 0
    for s in shards:
        sz = s.stat().st_size / 1024 / 1024
        total_size += sz
        print(f'  {s.name}: {sz:.2f} MB')
    print(f'  TOTAL: {total_size:.2f} MB across {len(shards)} shards')

# Sample articles
print('\nSample articles (first 5 from latest shard):')
if articles_dir.exists():
    shards = sorted(articles_dir.glob('articles_*.jsonl.gz'))
    if shards:
        with gzip.open(shards[-1], 'rt', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= 5: break
                art = json.loads(line)
                print(f'\n[{i+1}] {art["title"][:80]}')
                print(f'    Category: {art["category"]}')
                print(f'    Date:     {art["published_date"]}')
                print(f'    Length:   {art["char_count"]} chars')
                print(f'    Body:     {art["body_text"][:200]}...')

## Cell 6 — Utilities (optional)

**Reset checkpoint** (if you want to start completely fresh — deletes all progress!):
```python
# Uncomment to run:
# import shutil
# shutil.rmtree('/content/drive/MyDrive/urdu_corpus/arynews', ignore_errors=True)
# print('All data deleted. Re-run Cell 3 + Cell 4 to start over.')
```

**Retry failed URLs only** (after a complete run, try the failed ones again):
```python
# Uncomment to run:
# cp = json.load(open('/content/drive/MyDrive/urdu_corpus/arynews/checkpoint/progress.json'))
# failed_records = [{'url': u, 'category_name': 'unknown'} for u in cp['failed_urls']]
# # Reset failed list so they get retried
# cp['failed_urls'] = []
# json.dump(cp, open('/content/drive/MyDrive/urdu_corpus/arynews/checkpoint/progress.json', 'w'), ensure_ascii=False)
# state = fetch_all_articles(failed_records)
```